# MaterialForge: GRPO Training for Crystal Design

**Meta PyTorch OpenEnv Hackathon x Scaler School of Technology - Grand Finale**

This notebook trains a tool-using LLM to autonomously design crystalline structures inside the MaterialForge environment. The goal is not just to improve training reward, but to show a credible end-to-end RL story for the final round: environment interaction, reward improvement, held-out evaluation, qualitative rollouts, and optional artifact publishing.

| Component | Details |
|-----------|----------|
| **Environment** | MaterialForge - 8x8 atomic lattice, 4 atom species, 4 target physical properties |
| **Training Method** | GRPO (Group Relative Policy Optimization) via Hugging Face TRL |
| **Efficiency Layer** | Unsloth 4-bit QLoRA fine-tuning |
| **Primary Final-Round Goal** | Demonstrate that the trained policy acts better than a weak baseline on held-out scenarios |
| **Live Environment** | [ArshPathan/material_forge_env](https://huggingface.co/spaces/ArshPathan/material_forge_env) |

## What this notebook now produces
- A trained GRPO checkpoint (adapter-style save path)
- Reward and loss curves saved as plot artifacts
- Held-out evaluation comparing the trained policy against a random baseline
- A qualitative rollout transcript on a named scenario
- An optional Hugging Face publishing path for the trained adapters


## 1. Installation

In [ ]:
%%capture
!pip install -Uq unsloth
!pip install -Uq trl>=0.18.0 transformers datasets accelerate peft
!pip install -Uq "openenv-core[core]>=0.2.3" aiofiles openai python-dotenv
!pip install -Uq matplotlib jmespath

## 2. Clone Environment & Setup Paths

We clone the MaterialForge repository to access the environment code directly.
The same environment is hosted live at: https://huggingface.co/spaces/ArshPathan/material_forge_env

In [ ]:
import os

REPO_URL = "https://huggingface.co/spaces/ArshPathan/material_forge_env"
REPO_DIR = "MaterialForge"

if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print(f"{REPO_DIR}/ already exists, skipping clone.")

import sys
sys.path.insert(0, REPO_DIR)

print("Environment source ready.")

## 3. Verify Environment Works

Quick sanity check: reset the environment, take one action, confirm reward is computed.

In [ ]:
from environment.rubrics import HeuristicRewardRubric
from server.material_forge_env_environment import MaterialForgeEnvironment
from models import MaterialForgeAction

env = MaterialForgeEnvironment(rubric=HeuristicRewardRubric())
obs = env.reset(seed=42, difficulty="easy")
print(f"Target properties: {obs.target}")
print(f"Cost budget: {obs.cost_budget}")
print(f"Max steps: {obs.max_steps}")

action = MaterialForgeAction(action_type="place", row=3, col=3, atom="A")
obs = env.step(action)
print(f"\nAfter placing Metal at (3,3):")
print(f"  Reward: {obs.reward}")
print(f"  Phase: {obs.phase}")
print(f"  Properties: {obs.current_properties}")
print("\nEnvironment OK!")

## 4. TRL Environment Wrapper

TRL's `GRPOTrainer` with `environment_factory` requires a wrapper class that exposes **named tool methods** (not a generic `step()`). The trainer discovers these methods and presents them to the LLM as callable tools.

Our wrapper exposes three tools:
- `place_atom(row, col, atom)` — place a new atom on an empty cell
- `remove_atom(row, col)` — remove an existing atom
- `replace_atom(row, col, atom)` — swap an atom for a different species

Each tool internally calls the OpenEnv environment's standard `step()` method.

In [ ]:
import random
from typing import Optional

from environment.config import GRID_SIZE, PROPERTY_NAMES, ATOM_TYPES
from environment.rubrics import HeuristicRewardRubric
from models import MaterialForgeAction, MaterialForgeObservation
from server.material_forge_env_environment import MaterialForgeEnvironment


DIFFICULTIES = ["easy", "medium", "hard"]

# --- Curriculum Learning State ---
TOTAL_EPISODES_STARTED = 0


def get_curriculum_difficulty() -> str:
    """Select difficulty based on global episode progress."""
    global TOTAL_EPISODES_STARTED
    if TOTAL_EPISODES_STARTED < 25:
        difficulty = "easy"
    elif TOTAL_EPISODES_STARTED < 75:
        difficulty = "medium"
    else:
        difficulty = "hard"

    TOTAL_EPISODES_STARTED += 1
    return difficulty


class MaterialForgeTRLEnv:
    """TRL-compatible wrapper for MaterialForge crystal design."""

    def __init__(self):
        self.env = MaterialForgeEnvironment(rubric=HeuristicRewardRubric())
        self.reward = 0.0
        self.done = False
        self._obs = None
        self._best_reward = 0.0
        self._reward_history = []
        self._invalid_actions = 0
        self._total_actions = 0
        self._best_phase = "amorphous"
        self._best_rows_used = 0
        self._best_cols_used = 0
        self._no_improve_streak = 0

    def reset(self, **kwargs) -> str:
        seed = random.randint(0, 99999)
        difficulty = get_curriculum_difficulty()
        obs = self.env.reset(seed=seed, difficulty=difficulty)
        self._obs = obs
        self.reward = 0.0
        self.done = False
        self._best_reward = 0.0
        self._reward_history = []
        self._invalid_actions = 0
        self._total_actions = 0
        self._best_phase = obs.phase
        self._best_rows_used = 0
        self._best_cols_used = 0
        self._no_improve_streak = 0
        return self._format_observation(obs)

    def place_atom(self, row: int, col: int, atom: str) -> str:
        """Place an atom on an empty lattice cell.

        Args:
            row: Row index on the 8x8 lattice.
            col: Column index on the 8x8 lattice.
            atom: Atom species to place. Must be one of A, B, C, or P.

        Returns:
            Updated observation text describing the lattice, rewards, and property gaps.
        """
        return self._do_step("place", row, col, atom)

    def remove_atom(self, row: int, col: int) -> str:
        """Remove an atom from the lattice.

        Args:
            row: Row index on the 8x8 lattice.
            col: Column index on the 8x8 lattice.

        Returns:
            Updated observation text describing the lattice, rewards, and property gaps.
        """
        return self._do_step("remove", row, col, None)

    def replace_atom(self, row: int, col: int, atom: str) -> str:
        """Replace an existing atom with a different species.

        Args:
            row: Row index on the 8x8 lattice.
            col: Column index on the 8x8 lattice.
            atom: New atom species. Must be one of A, B, C, or P.

        Returns:
            Updated observation text describing the lattice, rewards, and property gaps.
        """
        return self._do_step("replace", row, col, atom)

    def _do_step(self, action_type: str, row: int, col: int, atom: Optional[str]) -> str:
        if self.done:
            raise ValueError("Episode is over. The crystal design session has ended.")

        self._total_actions += 1
        row = max(0, min(int(row), GRID_SIZE - 1))
        col = max(0, min(int(col), GRID_SIZE - 1))

        if atom is not None:
            atom = str(atom).upper().strip()
            if atom not in ATOM_TYPES:
                atom = "A"

        grid = self._obs.grid if self._obs else None
        if grid:
            cell = grid[row][col]
            if action_type == "place" and cell != ".":
                self._invalid_actions += 1
                self.reward = self._compute_episode_reward()
                return (
                    f"INVALID ACTION: Cannot place at ({row},{col}) - cell already contains '{cell}'.\n"
                    + self._format_observation(self._obs)
                )
            if action_type == "remove" and cell == ".":
                self._invalid_actions += 1
                self.reward = self._compute_episode_reward()
                return (
                    f"INVALID ACTION: Cannot remove from ({row},{col}) - cell is empty.\n"
                    + self._format_observation(self._obs)
                )
            if action_type == "replace" and cell == ".":
                self._invalid_actions += 1
                self.reward = self._compute_episode_reward()
                return (
                    f"INVALID ACTION: Cannot replace at ({row},{col}) - cell is empty. Use place_atom instead.\n"
                    + self._format_observation(self._obs)
                )
            if action_type == "replace" and cell == atom:
                self._invalid_actions += 1
                self.reward = self._compute_episode_reward()
                return (
                    f"INVALID ACTION: Cell ({row},{col}) already contains '{atom}'.\n"
                    + self._format_observation(self._obs)
                )

        action = MaterialForgeAction(action_type=action_type, row=row, col=col, atom=atom)
        obs = self.env.step(action)
        self._obs = obs

        step_reward = float(obs.reward or 0.0)
        self._reward_history.append(step_reward)

        improved = step_reward > (self._best_reward + 0.002)
        if improved:
            self._best_reward = step_reward
            self._best_rows_used, self._best_cols_used = self._count_rows_and_cols(obs.grid)
            self._best_phase = obs.phase
            self._no_improve_streak = 0
        else:
            self._no_improve_streak += 1

        self.reward = self._compute_episode_reward()

        if step_reward >= 0.90 and obs.phase in {"crystalline", "polycrystalline"} and self._total_actions >= 6:
            self.done = True
        elif self._best_reward >= 0.86 and self._no_improve_streak >= 6:
            self.done = True
        else:
            self.done = obs.done

        return self._format_observation(obs)

    def _count_rows_and_cols(self, grid: list[list[str]]) -> tuple[int, int]:
        rows_used = set()
        cols_used = set()
        for r, row in enumerate(grid):
            for c, cell in enumerate(row):
                if cell != ".":
                    rows_used.add(r)
                    cols_used.add(c)
        return len(rows_used), len(cols_used)

    def _compute_episode_reward(self) -> float:
        total = max(self._total_actions, 1)
        invalid_ratio = self._invalid_actions / total
        invalid_penalty = 0.35 * invalid_ratio

        row_spread = min(self._best_rows_used / 4.0, 1.0)
        col_spread = min(self._best_cols_used / 4.0, 1.0)
        spatial_bonus = 0.08 * (row_spread * col_spread)

        phase_bonus = 0.0
        if self._best_phase == "crystalline":
            phase_bonus = 0.10
        elif self._best_phase == "polycrystalline":
            phase_bonus = 0.04

        grace_actions = 8
        excess_actions = max(self._total_actions - grace_actions, 0)
        efficiency_penalty = 0.18 * min(excess_actions / 18.0, 1.0)

        stagnation_penalty = 0.10 * min(max(self._no_improve_streak - 3, 0) / 8.0, 1.0)

        final = self._best_reward + spatial_bonus + phase_bonus - invalid_penalty - efficiency_penalty - stagnation_penalty
        return max(min(final, 1.0), 0.0)

    def _format_observation(self, obs: MaterialForgeObservation) -> str:
        grid_lines = []
        for r, row in enumerate(obs.grid):
            grid_lines.append(f"  {r}: {' '.join(cell if cell != '.' else '.' for cell in row)}")
        grid_str = "\n".join(grid_lines)

        gaps = {}
        for prop in PROPERTY_NAMES:
            t = obs.target.get(prop, 0.0)
            c = obs.current_properties.get(prop, 0.0)
            gaps[prop] = round(t - c, 1)

        props_str = ", ".join(
            f"{p}={obs.current_properties.get(p, 0):.1f}/{obs.target.get(p, 0):.1f}(gap:{gaps[p]:+.1f})"
            for p in PROPERTY_NAMES
        )

        sb = obs.score_breakdown or {}
        stability = sb.get("structural_stability", 0.0)
        order = sb.get("lattice_order_index", 0.0)

        return (
            f"Step {obs.step_number}/{obs.max_steps} | Cost: {obs.total_cost:.0f}/{obs.cost_budget:.0f} | "
            f"Phase: {obs.phase} | StepReward: {float(obs.reward or 0.0):.4f} | EpisodeReward: {self.reward:.4f}\n"
            f"Actions: {self._total_actions} | Invalid: {self._invalid_actions} | NoImproveStreak: {self._no_improve_streak}\n"
            f"Properties: {props_str}\n"
            f"Stability: {stability:.3f} | Order: {order:.3f}\n"
            f"Guidance: Use as few tool calls as possible. Stop when reward is high and phase is ordered.\n"
            f"Grid (row: cols 0-7):\n{grid_str}"
        )


print("MaterialForgeTRLEnv defined with efficiency-aware shaping.")


### Test the wrapper

In [ ]:
test_env = MaterialForgeTRLEnv()
init_obs = test_env.reset()
print("=== Initial State ===")
print(init_obs)

print("\n=== After place_atom(3, 3, 'A') ===")
result = test_env.place_atom(3, 3, "A")
print(result)
print(f"\nReward stored: {test_env.reward:.4f}")
print(f"Done: {test_env.done}")

print("\n=== After place_atom(3, 4, 'B') ===")
result = test_env.place_atom(3, 4, "B")
print(result)
print(f"Reward: {test_env.reward:.4f}")

## 5. Reward Function

In [ ]:
def reward_func(environments, **kwargs) -> list[float]:
    """Episode reward that favors strong states reached with fewer, cleaner actions."""
    return [env._compute_episode_reward() for env in environments]

print("Reward function: best_reward + compactness + phase bonus - invalid - excess-tool - stagnation penalties")


## 6. Training Dataset

We intentionally keep the text dataset minimal: a fixed system/user prompt pair repeated across episodes. The diversity comes from the **environment**, not from supervised demonstrations. Each `reset()` produces a fresh target vector and new trajectory dynamics, which is exactly the behavior we want for RL-style post-training.

This is a good fit for the final round because it demonstrates that:
- the environment is doing real work,
- the reward function is shaping behavior,
- and the model is learning to act through repeated interaction rather than memorizing static traces.


In [ ]:
from datasets import Dataset

SYSTEM_PROMPT = """You are an expert materials scientist designing crystal structures on an 8x8 atomic lattice.

Your goal: place atoms to match target physical properties while staying within budget.

Available atom species:
- A (Metal): Primary contribution to hardness, cost 8 per atom
- B (Conductor): Primary contribution to conductivity, cost 6 per atom
- C (Ceramic): Primary contribution to thermal resistance, cost 4 per atom
- P (Polymer): Primary contribution to elasticity, cost 2 per atom

Strategy:
- START from the center of the grid (around row 3-4, col 3-4) and expand outward in all directions
- Build 2D clusters, NOT single rows or columns - atoms need neighbors in multiple directions for bonding
- Group same-type atoms together in 2x2 or 3x3 blocks for maximum bonding bonuses
- After each placement, READ the property gaps and choose the atom type that closes the LARGEST gap
- Aim for crystalline phase: fill a compact rectangular region, not scattered atoms
- Stay within the cost budget - use cheaper atoms (P=2, C=4) when budget is tight
- Use replace_atom to fix overshooting properties, remove_atom if over budget
- Make as FEW tool calls as possible; do not keep exploring once the structure is already strong
- If reward is high and the phase is ordered, stop calling tools
- Prefer decisive, compact trajectories rather than long wandering searches
- When tools are available, call exactly one tool at a time and use the observation after each tool call before deciding the next action

Use the tools place_atom, remove_atom, and replace_atom to build your crystal."""

USER_PROMPT = "Design a crystal lattice that matches the target material properties. Start from the center, grow compact 2D clusters, minimize unnecessary tool calls, and stop once the structure is already strong and ordered."

NUM_EPISODES = int(os.environ.get("NUM_EPISODES", 120))

dataset = Dataset.from_dict({
    "prompt": [
        [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": USER_PROMPT},
        ]
    ] * NUM_EPISODES
})

print(f"Dataset created: {len(dataset)} episodes")
print(f"Sample prompt roles: {[m['role'] for m in dataset[0]['prompt']]}")


## 7. Load Model with Unsloth

For the final round, the default recommendation is **`Qwen/Qwen3-1.7B`**. It is a more credible final-round starting point than `0.6B` while still being realistic on hackathon compute with 4-bit QLoRA.

Fallback guidance:
- If compute is tight: use `Qwen/Qwen3-0.6B`
- If extra compute is available: keep `1.7B` and increase episode count before moving to a much larger base model

The cell below also allows overriding the base model through `BASE_MODEL_NAME` so you can switch quickly without restructuring the notebook.


In [ ]:
import os
import torch
from unsloth import FastLanguageModel

MODEL_NAME = os.environ.get("BASE_MODEL_NAME", "Qwen/Qwen3-1.7B")
MAX_SEQ_LENGTH = int(os.environ.get("MAX_SEQ_LENGTH", 4096))
LOAD_IN_4BIT = os.environ.get("LOAD_IN_4BIT", "1") == "1"

model, tokenizer = FastLanguageModel.from_pretrained(
    MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=LOAD_IN_4BIT,
    dtype=None,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
)

# --- Fix Unsloth + TRL compatibility for Qwen3 tool calling ---
from pathlib import Path
import trl.chat_template_utils as _ctu

tokenizer.response_schema = {
    "x-regex": (
        r"^(?:<think>\n?(?:(?P<reasoning_content>.*?\S.*?)\n?|[\s]*)</think>\s*)?"
        r"(?P<content>.*?)(?:\n(?=<tool_call>))?"
        r"(?=(?:<tool_call>|<\|im_end\|>|$))"
        r"(?P<tool_calls>(?:<tool_call>.+?</tool_call>\s*)+)?\s*(?:<\|im_end\|>|$)"
    ),
    "type": "object",
    "properties": {
        "role": {"const": "assistant"},
        "content": {"type": "string"},
        "reasoning_content": {"type": "string"},
        "tool_calls": {
            "type": "array",
            "x-regex-iterator": r"<tool_call>\s*(.+?)\s*</tool_call>",
            "items": {
                "x-parser": "json",
                "x-parser-args": {"transform": "{type: 'function', function: @}"},
                "type": "object",
                "properties": {
                    "type": {"const": "function"},
                    "function": {
                        "type": "object",
                        "properties": {
                            "name": {"type": "string"},
                            "arguments": {"type": "object", "additionalProperties": {}},
                        },
                    },
                },
            },
        },
    },
}

_templates_dir = Path(_ctu.__file__).parent / "chat_templates"
tokenizer.chat_template = (_templates_dir / "qwen3_training.jinja").read_text()

print(f"Model loaded: {MODEL_NAME}")
print(f"Trainable params: {model.print_trainable_parameters()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Context length: {MAX_SEQ_LENGTH}")
print(f"4-bit loading: {LOAD_IN_4BIT}")
print("Chat template: TRL qwen3_training.jinja (prefix-preserving)")
print("Response schema: Qwen3 tool-call parser")


## 8. GRPO Training

The `GRPOTrainer` with `environment_factory` handles the full multi-turn loop:
1. Generates completions from the LLM
2. Parses tool calls (`place_atom`, `remove_atom`, `replace_atom`)
3. Executes them against the MaterialForge environment
4. Feeds observations back to the model
5. Collects rewards and updates the policy via GRPO

For final-round storytelling, the important thing is to keep the configuration explicit and reproducible. The cell below prints the exact runtime choices so the notebook doubles as a training record.


In [ ]:
import os
from trl import GRPOConfig, GRPOTrainer

os.environ.setdefault("TRL_EXPERIMENTAL_SILENCE", "1")

OUTPUT_DIR = os.environ.get("OUTPUT_DIR", "materialforge-grpo-checkpoints")
NUM_GENERATIONS = int(os.environ.get("NUM_GENERATIONS", 4))
MAX_COMPLETION_LENGTH = int(os.environ.get("MAX_COMPLETION_LENGTH", 1024))
LEARNING_RATE = float(os.environ.get("LEARNING_RATE", 5e-5))
GRAD_ACCUM_STEPS = int(os.environ.get("GRAD_ACCUM_STEPS", 4))
SAVE_STEPS = int(os.environ.get("SAVE_STEPS", max(25, NUM_EPISODES // 2)))
TEMPERATURE = float(os.environ.get("TEMPERATURE", 0.7))

training_args = GRPOConfig(
    output_dir=OUTPUT_DIR,

    # Generation
    num_generations=NUM_GENERATIONS,
    max_completion_length=MAX_COMPLETION_LENGTH,
    temperature=TEMPERATURE,

    # Optimization
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    num_train_epochs=1,

    # Logging
    logging_steps=1,
    log_completions=True,
    num_completions_to_print=1,

    # Tool calling
    chat_template_kwargs={"enable_thinking": False},

    # Saving
    save_steps=SAVE_STEPS,
    save_total_limit=2,
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=reward_func,
    train_dataset=dataset,
    args=training_args,
    environment_factory=MaterialForgeTRLEnv,
)

print("Trainer initialized. Starting GRPO training...")
print(f"  Base model: {MODEL_NAME}")
print(f"  Episodes: {NUM_EPISODES}")
print(f"  Generations per prompt: {training_args.num_generations}")
print(f"  Max completion length: {training_args.max_completion_length}")
print(f"  Temperature: {training_args.temperature}")
print(f"  Learning rate: {training_args.learning_rate}")
print(f"  Gradient accumulation: {training_args.gradient_accumulation_steps}")
print(f"  Save steps: {training_args.save_steps}")
print("  Runtime tuning: shorter completions + efficiency-aware reward shaping to reduce tool spam")


In [ ]:
train_result = trainer.train()
print(f"\nTraining complete!")
print(f"  Total steps: {trainer.state.global_step}")
print(f"  Final loss: {train_result.training_loss:.4f}")

## 9. Save Trained Model

This save path is intended to preserve a clean artifact for final-round judging and optional Hub publishing. In practice, for this notebook the most useful artifact is usually the **adapter checkpoint + tokenizer** rather than a heavyweight merged model.


In [ ]:
SAVE_DIR = os.environ.get("SAVE_DIR", "materialforge-grpo-trained")
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print(f"Model saved to {SAVE_DIR}/")


## 10. Training Curves and Log Artifacts

These plots are part of the required evidence for the final round. Besides the images, we also save the raw trainer log history so the results can be reused in the README, slides, or a Hugging Face post.


In [ ]:
import json
import matplotlib.pyplot as plt
from pathlib import Path

plot_dir = Path("plots")
plot_dir.mkdir(exist_ok=True)

log_history = trainer.state.log_history
with (plot_dir / "trainer_log_history.json").open("w", encoding="utf-8") as f:
    json.dump(log_history, f, indent=2)

reward_steps, rewards = [], []
loss_steps, losses = [], []

for i, entry in enumerate(log_history):
    step = entry.get("step", i)
    reward_key = next(
        (k for k, v in entry.items() if "reward" in k.lower() and isinstance(v, (int, float))),
        None,
    )
    if reward_key is not None:
        reward_steps.append(step)
        rewards.append(float(entry[reward_key]))
    if "loss" in entry and isinstance(entry["loss"], (int, float)):
        loss_steps.append(step)
        losses.append(float(entry["loss"]))


def bucket_mean(steps, values, bucket_size=5):
    buckets = {}
    for step, value in zip(steps, values):
        bucket = (int(step) // bucket_size) * bucket_size
        buckets.setdefault(bucket, []).append(float(value))
    bucket_steps = sorted(buckets)
    bucket_values = [sum(buckets[b]) / len(buckets[b]) for b in bucket_steps]
    return bucket_steps, bucket_values

bucket_size = 5
reward_bucket_steps, reward_bucket_values = bucket_mean(reward_steps, rewards, bucket_size=bucket_size)
loss_bucket_steps, loss_bucket_values = bucket_mean(loss_steps, losses, bucket_size=bucket_size) if losses else ([], [])

# --- Plot 1: Reward over training, bucketed by 5 steps ---
fig, ax = plt.subplots(figsize=(10, 5))
if rewards:
    ax.plot(reward_steps, rewards, color="#9bbcf5", linewidth=1.2, alpha=0.25, label="Raw reward")
    ax.plot(reward_bucket_steps, reward_bucket_values, color="#1d4ed8", linewidth=2.8, marker="o", label=f"Bucketed mean ({bucket_size}-step bins)")
ax.set_xlabel("Training Step Bucket", fontsize=12)
ax.set_ylabel("Reward", fontsize=12)
ax.set_title("MaterialForge GRPO Training - Reward Curve (5-Step Mean)", fontsize=14, fontweight="bold")
ax.set_xticks(reward_bucket_steps)
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(plot_dir / "reward_curve.png", dpi=150)
plt.show()
print("Saved: plots/reward_curve.png")

# --- Plot 2: Loss over training, bucketed by 5 steps ---
if losses:
    fig2, ax2 = plt.subplots(figsize=(10, 5))
    ax2.plot(loss_steps, losses, color="#86efac", linewidth=1.2, alpha=0.25, label="Raw loss")
    ax2.plot(loss_bucket_steps, loss_bucket_values, color="#15803d", linewidth=2.8, marker="o", label=f"Bucketed mean ({bucket_size}-step bins)")
    ax2.set_xlabel("Training Step Bucket", fontsize=12)
    ax2.set_ylabel("Loss", fontsize=12)
    ax2.set_title("MaterialForge GRPO Training - Loss Curve (5-Step Mean)", fontsize=14, fontweight="bold")
    ax2.set_xticks(loss_bucket_steps)
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    fig2.tight_layout()
    fig2.savefig(plot_dir / "loss_curve.png", dpi=150)
    plt.show()
    print("Saved: plots/loss_curve.png")

print("Saved: plots/trainer_log_history.json")


## 11. Held-Out Evaluation

Training reward alone is not enough for the final round. We need to show how the trained policy behaves on explicit evaluation episodes that were **not** just summarized from the trainer logs.

This section evaluates the trained model on a small held-out suite of named and random scenarios, compares it against a random baseline, and saves presentation-ready plots.


In [ ]:
import json
import random as rng
import re
import statistics
from collections import Counter

EVAL_SYSTEM_PROMPT = SYSTEM_PROMPT + """

Tool-calling rules for evaluation:
- Return exactly one tool call per turn.
- Do not return prose before the tool call.
- Use this exact wrapper format:
  <tool_call>{"name": "place_atom", "arguments": {"row": 3, "col": 3, "atom": "A"}}</tool_call>
- Allowed tool names: place_atom, remove_atom, replace_atom
"""


def format_observation_for_eval(obs):
    gaps = {p: round(obs.target[p] - obs.current_properties.get(p, 0.0), 2) for p in PROPERTY_NAMES}
    grid_lines = [f"  {r}: {' '.join(row)}" for r, row in enumerate(obs.grid)]
    sb = obs.score_breakdown or {}
    return (
        f"Step {obs.step_number}/{obs.max_steps}\n"
        f"Phase: {obs.phase}\n"
        f"Cost: {obs.total_cost:.0f}/{obs.cost_budget:.0f}\n"
        f"Target: {obs.target}\n"
        f"Current: {obs.current_properties}\n"
        f"Gaps: {gaps}\n"
        f"Stability: {sb.get('structural_stability', 0.0):.3f} | Order: {sb.get('lattice_order_index', 0.0):.3f}\n"
        f"Grid:\n" + "\n".join(grid_lines)
    )


def extract_tool_call(text):
    if not text:
        return None
    cleaned = text.strip()
    if "```" in cleaned:
        parts = cleaned.split("```")
        cleaned = "\n".join(parts[1:-1] or parts)
    match = re.search(r"<tool_call>\s*(\{.*?\})\s*</tool_call>", cleaned, flags=re.S)
    payload_text = match.group(1) if match else cleaned
    start = payload_text.find("{")
    end = payload_text.rfind("}") + 1
    if start == -1 or end <= start:
        return None
    try:
        payload = json.loads(payload_text[start:end])
        if "function" in payload:
            payload = payload["function"]
        name = payload.get("name")
        arguments = payload.get("arguments", {})
        if isinstance(arguments, str):
            arguments = json.loads(arguments)
        if not isinstance(arguments, dict):
            return None
        return {"name": name, "arguments": arguments}
    except Exception:
        return None


def generate_tool_response(model, tokenizer, messages, max_new_tokens=256):
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=0.0,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=False)


def execute_tool_call(env, tool_name, arguments):
    row = int(arguments.get("row", 0))
    col = int(arguments.get("col", 0))
    atom = arguments.get("atom")
    if atom is not None:
        atom = str(atom).upper().strip()

    if tool_name == "place_atom":
        action = MaterialForgeAction(action_type="place", row=row, col=col, atom=atom)
    elif tool_name == "remove_atom":
        action = MaterialForgeAction(action_type="remove", row=row, col=col, atom=None)
    elif tool_name == "replace_atom":
        action = MaterialForgeAction(action_type="replace", row=row, col=col, atom=atom)
    else:
        raise ValueError(f"Unsupported tool: {tool_name}")

    return env.step(action)


def episode_success(obs):
    tol = float((obs.metadata or {}).get("tolerance", 0.0))
    within = all(abs(obs.current_properties.get(p, 0.0) - obs.target.get(p, 0.0)) <= tol for p in PROPERTY_NAMES)
    return bool(obs.done and within and obs.phase in {"crystalline", "polycrystalline"})


def run_trained_episode(model, tokenizer, task, max_tool_steps=None):
    env = MaterialForgeEnvironment(rubric=HeuristicRewardRubric())
    obs = env.reset(seed=task["seed"], difficulty=task["difficulty"], scenario_name=task.get("scenario_name"))
    max_tool_steps = max_tool_steps or obs.max_steps

    messages = [
        {"role": "system", "content": EVAL_SYSTEM_PROMPT},
        {"role": "user", "content": USER_PROMPT + "\n\nCurrent observation:\n" + format_observation_for_eval(obs)},
    ]

    rewards = []
    transcript = []
    parse_failures = 0

    while not obs.done and obs.step_number < max_tool_steps:
        raw = generate_tool_response(model, tokenizer, messages)
        call = extract_tool_call(raw)
        if call is None or call.get("name") not in {"place_atom", "remove_atom", "replace_atom"}:
            parse_failures += 1
            transcript.append({
                "step": obs.step_number + 1,
                "error": "tool_parse_failure",
                "raw": raw[:400],
            })
            break

        try:
            obs = execute_tool_call(env, call["name"], call["arguments"])
        except Exception as exc:
            parse_failures += 1
            transcript.append({
                "step": obs.step_number + 1,
                "error": f"tool_execution_error: {exc}",
                "raw": raw[:400],
            })
            break

        reward = float(obs.reward or 0.0)
        rewards.append(reward)
        transcript.append({
            "step": obs.step_number,
            "tool": call["name"],
            "arguments": call["arguments"],
            "reward": reward,
            "phase": obs.phase,
            "cost": obs.total_cost,
        })
        messages.append({"role": "assistant", "content": raw})
        messages.append({"role": "user", "content": "Environment observation after tool execution:\n" + format_observation_for_eval(obs)})

    return {
        "label": task["label"],
        "difficulty": task["difficulty"],
        "scenario_name": task.get("scenario_name") or "random",
        "seed": task["seed"],
        "mean_reward": sum(rewards) / len(rewards) if rewards else 0.0,
        "best_reward": max(rewards) if rewards else 0.0,
        "steps": obs.step_number,
        "phase": obs.phase,
        "success": episode_success(obs),
        "parse_failures": parse_failures,
        "transcript": transcript,
        "final_observation": obs,
    }


def random_valid_action(obs):
    coords = [(r, c) for r in range(8) for c in range(8)]
    rng.shuffle(coords)
    for r, c in coords:
        cell = obs.grid[r][c]
        if cell == ".":
            return MaterialForgeAction(action_type="place", row=r, col=c, atom=rng.choice(["A", "B", "C", "P"]))
    for r, c in coords:
        cell = obs.grid[r][c]
        if cell != ".":
            replacement = rng.choice([a for a in ["A", "B", "C", "P"] if a != cell])
            return MaterialForgeAction(action_type="replace", row=r, col=c, atom=replacement)
    return MaterialForgeAction(action_type="remove", row=0, col=0, atom=None)


def run_random_episode(task):
    env = MaterialForgeEnvironment(rubric=HeuristicRewardRubric())
    obs = env.reset(seed=task["seed"], difficulty=task["difficulty"], scenario_name=task.get("scenario_name"))
    rewards = []

    while not obs.done and obs.step_number < obs.max_steps:
        action = random_valid_action(obs)
        obs = env.step(action)
        rewards.append(float(obs.reward or 0.0))

    return {
        "label": task["label"],
        "difficulty": task["difficulty"],
        "scenario_name": task.get("scenario_name") or "random",
        "seed": task["seed"],
        "mean_reward": sum(rewards) / len(rewards) if rewards else 0.0,
        "best_reward": max(rewards) if rewards else 0.0,
        "steps": obs.step_number,
        "phase": obs.phase,
        "success": episode_success(obs),
        "parse_failures": 0,
        "transcript": [],
        "final_observation": obs,
    }


def summarize_runs(runs):
    return {
        "mean_reward": statistics.mean(run["mean_reward"] for run in runs),
        "mean_best_reward": statistics.mean(run["best_reward"] for run in runs),
        "success_rate": sum(1 for run in runs if run["success"]) / len(runs),
        "mean_steps": statistics.mean(run["steps"] for run in runs),
        "phase_counts": dict(Counter(run["phase"] for run in runs)),
        "parse_failures": sum(run["parse_failures"] for run in runs),
    }


In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt

EVAL_TASKS = [
    {"label": "easy-random", "difficulty": "easy", "seed": 101},
    {"label": "medium-random", "difficulty": "medium", "seed": 202},
    {"label": "hard-random", "difficulty": "hard", "seed": 303},
    {"label": "diamond-like", "difficulty": "medium", "seed": 42, "scenario_name": "diamond-like"},
    {"label": "conductor", "difficulty": "medium", "seed": 84, "scenario_name": "conductor"},
    {"label": "heat-shield", "difficulty": "medium", "seed": 126, "scenario_name": "heat-shield"},
]

trained_eval_runs = [run_trained_episode(model, tokenizer, task) for task in EVAL_TASKS]
random_eval_runs = [run_random_episode(task) for task in EVAL_TASKS]

trained_summary = summarize_runs(trained_eval_runs)
random_summary = summarize_runs(random_eval_runs)

print("Held-out evaluation complete.\n")
print("Per-task results:")
for trained_run, random_run in zip(trained_eval_runs, random_eval_runs):
    print(
        f"- {trained_run['label']}: "
        f"trained mean={trained_run['mean_reward']:.4f}, best={trained_run['best_reward']:.4f}, phase={trained_run['phase']}, success={trained_run['success']} | "
        f"random mean={random_run['mean_reward']:.4f}, best={random_run['best_reward']:.4f}, phase={random_run['phase']}, success={random_run['success']}"
    )

print("\nAggregate summary:")
print(f"Trained mean reward: {trained_summary['mean_reward']:.4f}")
print(f"Random mean reward:  {random_summary['mean_reward']:.4f}")
print(f"Trained success rate: {trained_summary['success_rate'] * 100:.1f}%")
print(f"Random success rate:  {random_summary['success_rate'] * 100:.1f}%")
print(f"Trained phase counts: {trained_summary['phase_counts']}")
print(f"Random phase counts:  {random_summary['phase_counts']}")
print(f"Trained parse failures: {trained_summary['parse_failures']}")

# Save evaluation JSON for README/slides reuse.
eval_payload = {
    "model_name": MODEL_NAME,
    "tasks": EVAL_TASKS,
    "trained_runs": [
        {
            k: v for k, v in run.items()
            if k not in {"final_observation"}
        }
        for run in trained_eval_runs
    ],
    "random_runs": [
        {
            k: v for k, v in run.items()
            if k not in {"final_observation"}
        }
        for run in random_eval_runs
    ],
    "trained_summary": trained_summary,
    "random_summary": random_summary,
}
with Path("plots/heldout_evaluation.json").open("w", encoding="utf-8") as f:
    json.dump(eval_payload, f, indent=2)

# Comparison plot
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].bar(["Random", "Trained"], [random_summary["mean_reward"], trained_summary["mean_reward"]], color=["#ea4335", "#4285f4"])
axes[0].set_ylabel("Mean Reward")
axes[0].set_title("Held-Out Mean Reward")
axes[0].grid(axis="y", alpha=0.3)

axes[1].bar(["Random", "Trained"], [random_summary["success_rate"] * 100, trained_summary["success_rate"] * 100], color=["#ea4335", "#4285f4"])
axes[1].set_ylabel("Success Rate (%)")
axes[1].set_title("Held-Out Success Rate")
axes[1].grid(axis="y", alpha=0.3)

fig.tight_layout()
fig.savefig("plots/heldout_comparison.png", dpi=150)
plt.show()
print("Saved: plots/heldout_comparison.png")
print("Saved: plots/heldout_evaluation.json")


## 12. Qualitative Trained Rollout Demo

A final-round notebook should show not only aggregate metrics, but also at least one readable rollout transcript. The cell below runs the trained model on a named scenario and prints the action trace, reward progression, and final state.


In [ ]:
print("Running qualitative demo on the diamond-like scenario...\n")

demo_task = {"label": "demo-diamond-like", "difficulty": "medium", "seed": 42, "scenario_name": "diamond-like"}
demo_run = run_trained_episode(model, tokenizer, demo_task)
demo_obs = demo_run["final_observation"]

print("=" * 72)
print("TRAINED AGENT QUALITATIVE DEMO")
print("=" * 72)
print(f"Scenario: {demo_run['scenario_name']} | Difficulty: {demo_run['difficulty']} | Seed: {demo_run['seed']}")
print(f"Steps taken: {demo_run['steps']}")
print(f"Mean reward: {demo_run['mean_reward']:.4f}")
print(f"Best reward: {demo_run['best_reward']:.4f}")
print(f"Final phase: {demo_run['phase']}")
print(f"Success: {demo_run['success']}")
print(f"Parse failures: {demo_run['parse_failures']}")
print("\nTranscript:")
for item in demo_run['transcript']:
    if 'tool' in item:
        print(
            f"  Step {item['step']:02d}: {item['tool']}({item['arguments']}) -> "
            f"reward={item['reward']:.4f}, phase={item['phase']}, cost={item['cost']:.0f}"
        )
    else:
        print(f"  Step {item['step']:02d}: {item['error']}")
        if item.get('raw'):
            print(f"    Raw model output: {item['raw'][:240]}")

print("\nFinal observation:")
print(format_observation_for_eval(demo_obs))


## 13. Optional: Publish the Trained Adapters to Hugging Face

Publishing the trained model is **optional** for this hackathon. The environment itself should be on Hugging Face Spaces, but uploading the trained adapter checkpoint is only a bonus for reproducibility and credibility.

The safest practical choice here is to publish the **adapter-style checkpoint** from `SAVE_DIR` rather than trying to merge a full model inside the notebook.


In [ ]:
HF_MODEL_REPO = os.environ.get("HF_MODEL_REPO", "")
PUBLISH_TO_HF = os.environ.get("PUBLISH_TO_HF", "0") == "1"

print("Model publishing is optional. The environment must be on HF Spaces; the trained model does not have to be.")

if PUBLISH_TO_HF and HF_MODEL_REPO:
    print(f"Pushing adapters and tokenizer to {HF_MODEL_REPO} ...")
    tokenizer.push_to_hub(HF_MODEL_REPO)
    model.push_to_hub(HF_MODEL_REPO)
    print("Push complete.")
else:
    print("Skipping Hub push.")
    print("To publish later, set:")
    print("  os.environ['HF_MODEL_REPO'] = '<username>/materialforge-grpo-qwen3-1.7b'")
    print("  os.environ['PUBLISH_TO_HF'] = '1'")


## 14. Final Summary

This notebook now covers the full final-round story:
- environment sanity check,
- TRL tool-calling wrapper,
- GRPO training,
- saved training curves,
- held-out evaluation,
- qualitative rollout evidence,
- and optional Hugging Face publishing.

The code cell below prints a compact summary you can reuse in the README, slides, or speaking notes.


In [ ]:
summary_lines = [
    "=" * 72,
    "MATERIALFORGE FINAL-ROUND NOTEBOOK SUMMARY",
    "=" * 72,
    f"Base model: {MODEL_NAME}",
    f"Episodes: {NUM_EPISODES}",
    f"Checkpoint directory: {SAVE_DIR}",
    f"Random held-out mean reward:  {random_summary['mean_reward']:.4f}",
    f"Trained held-out mean reward: {trained_summary['mean_reward']:.4f}",
    f"Random success rate:  {random_summary['success_rate'] * 100:.1f}%",
    f"Trained success rate: {trained_summary['success_rate'] * 100:.1f}%",
    f"Random phase counts:  {random_summary['phase_counts']}",
    f"Trained phase counts: {trained_summary['phase_counts']}",
    f"Trained parse failures: {trained_summary['parse_failures']}",
    "Artifacts:",
    "  - plots/reward_curve.png",
    "  - plots/loss_curve.png",
    "  - plots/heldout_comparison.png",
    "  - plots/heldout_evaluation.json",
    "  - materialforge-grpo-trained/",
]
print("\n".join(summary_lines))
